[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RogerioLS/production-ai-systems/blob/main/docs/notebooks/a_llm_basics/tokenization_playground.ipynb)

# 📊 Tokenization & Statistical Compression Playground

Welcome to the interactive playground for **LAB-01 (Tokenization Compression)**.

This notebook allows you to:
1. Interactively tokenize text using different algorithms (BPE vs WordPiece).
2. Compare models like GPT-4o (`o200k_base`), GPT-4 (`cl100k_base`), GPT-2 (`gpt2`), and BERT (`bert-uncased`).
3. Compute token footprint, compression ratios, and cost implications across different languages and structured data domains.

---

### 0. Environment Setup
If you are running in Google Colab, this cell will automatically clone the repository and install all required dependencies (like `tiktoken` and `transformers`). If running locally, it does nothing.

In [ ]:
import os
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("🤖 Running in Google Colab. Setting up environment...")
    !pip install -q numpy tiktoken transformers loguru
    
    # Clone repository to access custom local source modules
    if not os.path.exists("production-ai-systems"):
        !git clone https://github.com/RogerioLS/production-ai-systems.git
    
    sys.path.append("production-ai-systems")
    print("🎉 Colab environment successfully initialized!")
else:
    print("💻 Running in local environment.")

### 1. Setup & Imports
Let's import our custom libraries and standard requirements. Make sure you are running this in your active Conda environment.

In [ ]:
# Ensure project root is in the path when running locally
if not IN_COLAB:
    sys.path.append(os.path.abspath("../../../"))

from projects.a_llm_basics.src.tokenizer_math import (
    HuggingFaceTokenizer,
    TiktokenTokenizer,
    TokenizationAnalyzer,
)

print("✅ Tokenizer modules imported successfully!")

### 2. Loading Tokenizers
We load wraps for different models representing different vocabularies and base algorithms:
- **tiktoken GPT-4o (`o200k_base`):** Expanded BPE vocabulary of 200,000 tokens.
- **tiktoken GPT-4 (`cl100k_base`):** Standard BPE vocabulary of 100,000 tokens.
- **tiktoken GPT-2 (`gpt2`):** Legacy BPE vocabulary of 50,000 tokens.
- **Hugging Face BERT (`bert-uncased`):** WordPiece algorithm vocabulary of 30,000 tokens.

In [ ]:
tokenizers = [
    TiktokenTokenizer("o200k_base"),
    TiktokenTokenizer("cl100k_base"),
    TiktokenTokenizer("gpt2"),
    HuggingFaceTokenizer("google-bert/bert-base-uncased")
]
analyzer = TokenizationAnalyzer(tokenizers)
print("✅ All tokenizers initialized!")

### 3. Interactive Text Tokenization Analyzer
Enter a custom string below to benchmark the tokenizers. Look at the token count and the compression ratio (UTF-8 Bytes / Token).

*A higher compression ratio means the tokenizer compresses the text into fewer tokens, translating directly to lower API costs and shorter context windows.*

In [ ]:
# Enter your custom text to test below
custom_text = "IA para produção exige robustez, pipelines claros e monitoramento constante em ambientes de nuvem. 🚀"

print(f"Original text: {custom_text}")
print(f"Total length: {len(custom_text)} characters | {len(custom_text.encode('utf-8'))} UTF-8 bytes\n")

# Perform analysis
results = analyzer.analyze_corpus({"interactive_input": custom_text})

print("| Tokenizer Name | Tokens Count | Compression Ratio (Bytes/Token) |")
print("|----------------|--------------|---------------------------------|")
for t_name, metrics in results["interactive_input"].items():
    print(f"| {t_name:<14} | {metrics['token_count']:<12} | {metrics['compression_ratio']:<30.3f} |")

### 4. Code & Character Footprint (Visual Comparison)
Let's see what the tokens actually look like. We will print the list of token IDs and their decoded string representations to understand where BPE splits text (like splitting Portuguese syllables, structured JSON, or finance digits).

In [ ]:
test_word = "anticonstitucionalmente"
print(f"Comparing tokenization segments for word: '{test_word}'\n")

for t in tokenizers:
    ids = t.encode(test_word)
    # Decode each token individually to visualize splits
    decoded_parts = []
    for token_id in ids:
        try:
            part = t.decode([token_id])
            decoded_parts.append(part)
        except:
            decoded_parts.append("")
            
    print(f"Tokenizer: {t.name:<15} | Tokens: {len(ids)} | Split: {decoded_parts}")

### 5. Domain Corpus Analysis
Let's run a benchmark across various specialized text corpora to verify general tokenizer performance trends.

In [ ]:
corpus = {
    "English (Plain)": "Large Language Models learn statistical distributions of text to predict next tokens.",
    "Portuguese (PT-BR)": "Modelos de Linguagem de Grande Porte aprendem distribuições estatísticas de texto para prever os próximos tokens.",
    "Structured JSON": '{"model": "gpt-4o", "parameters": {"temperature": 0.7, "top_p": 0.9}, "tags": ["llm", "prod"]}',
    "Numeric / Financial": "$1,245,678.90 + $98,765.43 - $43,210.00 = $1,301,234.33",
    "Emojis & Special": "🤖🚀🧠✨🔥🌈💻🛠️"
}

analysis = analyzer.analyze_corpus(corpus)

for domain, tok_results in analysis.items():
    print(f"\nDomain: {domain}")
    print(f"UTF-8 Bytes: {len(corpus[domain].encode('utf-8'))}")
    print("-" * 50)
    for tok_name, metrics in tok_results.items():
        print(f"  - {tok_name:<14} : {metrics['token_count']} tokens ({metrics['compression_ratio']:.3f} B/T)")